<a href="https://colab.research.google.com/github/muzazr/favera-model/blob/main/03_candidate_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:




















from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [49]:
!pip install -q \
    polars \
    pyarrow \
    pandas \
    numpy \
    scikit-learn \
    lightgbm \
    shap \
    matplotlib \
    joblib

In [50]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/favera/model/transaction-anomaly"
)

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"

for directory in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    MODEL_DIR,
    OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

In [51]:
import random
import joblib
import numpy as np
import pandas as pd
import polars as pl

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    precision_score,
    recall_score,
)
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [52]:
DATA_PATH = RAW_DATA_DIR / "momtsim_v1.csv"

lazy_df = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    ignore_errors=True
)

print(lazy_df.collect_schema())

sample_df = lazy_df.head(5).collect()
sample_df

row_count = lazy_df.select(pl.len()).collect().item()
print(f"jumlah transaksi: {row_count:,}")

Schema({'step': Int64, 'transactionType': String, 'amount': Float64, 'initiator': Int64, 'oldBalInitiator': Float64, 'newBalInitiator': Float64, 'recipient': String, 'oldBalRecipient': Float64, 'newBalRecipient': Float64, 'isFraud': Int64})
jumlah transaksi: 1,720,181


## 2. Memahami Struktur Dataset

In [53]:
from IPython.display import display, Markdown
import polars as pl

schema = lazy_df.collect_schema()
columns = schema.names()

schema_df = pl.DataFrame({
    "column": columns,
    "data_type": [str(schema[col]) for col in columns]
})

display(Markdown(
    f"Dataset memiliki **{row_count:,} baris** dan "
    f"**{len(columns)} kolom**."
))

display(schema_df)

Dataset memiliki **1,720,181 baris** dan **10 kolom**.

column,data_type
str,str
"""step""","""Int64"""
"""transactionType""","""String"""
"""amount""","""Float64"""
"""initiator""","""Int64"""
"""oldBalInitiator""","""Float64"""
"""newBalInitiator""","""Float64"""
"""recipient""","""String"""
"""oldBalRecipient""","""Float64"""
"""newBalRecipient""","""Float64"""


In [54]:
preview_df = lazy_df.head(10).collect()

display(preview_df)

EDA_SAMPLE_SIZE = 100_000

eda_df = lazy_df.head(EDA_SAMPLE_SIZE).collect(
    engine="streaming"
)

display(Markdown(
    f"Sampel EDA menggunakan **{eda_df.height:,} baris**."
))

step,transactionType,amount,initiator,oldBalInitiator,newBalInitiator,recipient,oldBalRecipient,newBalRecipient,isFraud
i64,str,f64,i64,f64,f64,str,f64,f64,i64
0,"""TRANSFER""",19824.96,4537027967639631,187712.18,167887.22,"""4875702729424478""",8.31,19833.27,1
0,"""PAYMENT""",598.97,4296267625767470,8.92,8.92,"""25-0000401""",0.0,0.0,0
0,"""PAYMENT""",545.85,4178224023847746,93.6,-452.25,"""13-0001587""",0.0,545.85,0
0,"""TRANSFER""",19847.01,4178224023847746,-452.25,-20299.26,"""4096920916696293""",4011.72,23858.74,1
0,"""PAYMENT""",546.89,4779013371563747,159148.76,158601.88,"""75-0003564""",0.0,546.89,0
0,"""TRANSFER""",16973.75,4779013371563747,158601.88,141628.12,"""4593247015015219""",54.33,17028.08,1
0,"""PAYMENT""",659.02,4479448076398077,261950.48,261291.46,"""11-0003866""",0.0,659.02,0
0,"""TRANSFER""",21103.99,4479448076398077,261291.46,240187.47,"""4649550475321678""",19.63,21123.63,1
0,"""PAYMENT""",648.89,4587942659105635,4808.94,4160.05,"""64-0008405""",0.0,648.89,0


Sampel EDA menggunakan **100,000 baris**.

In [55]:
null_summary = (
    lazy_df
    .select([
        pl.col(col).null_count().alias(col)
        for col in columns
    ])
    .collect(engine="streaming")
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["null_count"]
    )
    .with_columns(
        (
            pl.col("null_count") / row_count * 100
        ).round(4).alias("null_percentage")
    )
    .sort("null_count", descending=True)
)

columns_with_null = null_summary.filter(
    pl.col("null_count") > 0
)

if columns_with_null.height == 0:
    display(Markdown("**Tidak ditemukan missing value.**"))
else:
    display(columns_with_null)

**Tidak ditemukan missing value.**

In [56]:
sample_schema = eda_df.schema

numeric_cols = [
    col
    for col, dtype in sample_schema.items()
    if dtype.is_numeric()
]

string_cols = [
    col
    for col, dtype in sample_schema.items()
    if dtype == pl.String
]

print("Kolom numerik:", numeric_cols)
print("Kolom teks:", string_cols)



Kolom numerik: ['step', 'amount', 'initiator', 'oldBalInitiator', 'newBalInitiator', 'oldBalRecipient', 'newBalRecipient', 'isFraud']
Kolom teks: ['transactionType', 'recipient']


In [57]:
if numeric_cols:
    numeric_summary = eda_df.select(
        numeric_cols
    ).describe()

    display(numeric_summary)
else:
    display(Markdown(
        "**Tidak ditemukan kolom numerik pada hasil parsing awal.**"
    ))

statistic,step,amount,initiator,oldBalInitiator,newBalInitiator,oldBalRecipient,newBalRecipient,isFraud
str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",5.56527,56655.657033,4.5010e15,287459.142625,332396.509199,7835.952872,12129.0505,0.11091
"""std""",2.983908,80062.006032,2.9027e14,569433.351835,601407.309418,37099.060757,42482.275094,0.314022
"""min""",0.0,1.36,4.0001e15,-199997.11,-199997.11,-149882.02,-135727.98,0.0
"""25%""",3.0,598.6,4.2472e15,18.44,20.4,534.48,1159.69,0.0
"""50%""",7.0,17572.83,4.5055e15,37481.59,83645.61,1735.43,2377.61,0.0
"""75%""",8.0,91043.64,4.7535e15,340451.41,425667.37,3303.1,4703.7,0.0
"""max""",9.0,1308624.7,4.9999e15,5.5312e6,5662492.3,1.5432e6,2.6180e6,1.0


In [58]:
if string_cols:
    unique_summary = pl.DataFrame({
        "column": string_cols,
        "unique_count_sample": [
            eda_df[col].n_unique()
            for col in string_cols
        ]
    }).sort(
        "unique_count_sample",
        descending=True
    )

    display(unique_summary)

column,unique_count_sample
str,i64
"""recipient""",10090
"""transactionType""",5


In [59]:
duplicate_count_sample = int(
    eda_df.is_duplicated().sum()
)

duplicate_percentage = (
    duplicate_count_sample / eda_df.height * 100
)

display(Markdown(
    f"""
**Duplikasi identik pada sampel:**
- Jumlah: **{duplicate_count_sample:,} baris**
- Persentase: **{duplicate_percentage:.4f}%**
"""
))


**Duplikasi identik pada sampel:**
- Jumlah: **0 baris**
- Persentase: **0.0000%**


In [60]:
target_keywords = [
    "fraud",
    "label",
    "class",
    "target",
    "anomaly"
]

time_keywords = [
    "date",
    "time",
    "timestamp",
    "step"
]

target_candidates = [
    col for col in columns
    if any(keyword in col.lower() for keyword in target_keywords)
]

time_candidates = [
    col for col in columns
    if any(keyword in col.lower() for keyword in time_keywords)
]

display(Markdown(
    f"""
### Kandidat Kolom Penting

**Kandidat label:** `{target_candidates}`

**Kandidat waktu:** `{time_candidates}`
"""
))


### Kandidat Kolom Penting

**Kandidat label:** `['isFraud']`

**Kandidat waktu:** `['step']`


In [61]:
for target_col in target_candidates:
    target_distribution = (
        lazy_df
        .group_by(target_col)
        .agg(
            pl.len().alias("count")
        )
        .with_columns(
            (
                pl.col("count") / row_count * 100
            ).round(4).alias("percentage")
        )
        .sort("count", descending=True)
        .collect(engine="streaming")
    )

    display(Markdown(
        f"### Distribusi `{target_col}`"
    ))
    display(target_distribution)

### Distribusi `isFraud`

isFraud,count,percentage
i64,u32,f64
0,1544663,89.7965
1,175518,10.2035


In [62]:
import re


def normalize_column_name(name: str) -> str:
    """Mengubah nama kolom menjadi format snake_case."""
    name = name.strip()
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)
    return name.strip("_").lower()


rename_map = {
    col: normalize_column_name(col)
    for col in columns
}

clean_lf = lazy_df.rename(rename_map)

display(
    pl.DataFrame({
        "original_column": list(rename_map.keys()),
        "clean_column": list(rename_map.values())
    })
)

original_column,clean_column
str,str
"""step""","""step"""
"""transactionType""","""transaction_type"""
"""amount""","""amount"""
"""initiator""","""initiator"""
"""oldBalInitiator""","""old_bal_initiator"""
"""newBalInitiator""","""new_bal_initiator"""
"""recipient""","""recipient"""
"""oldBalRecipient""","""old_bal_recipient"""
"""newBalRecipient""","""new_bal_recipient"""


In [63]:
clean_schema = clean_lf.collect_schema()

clean_string_cols = [
    col
    for col, dtype in clean_schema.items()
    if dtype == pl.String
]

NULL_TOKENS = [
    "",
    "na",
    "n/a",
    "null",
    "none",
    "-"
]

clean_lf = clean_lf.with_columns([
    pl.when(
        pl.col(col)
        .str.strip_chars()
        .str.to_lowercase()
        .is_in(NULL_TOKENS)
    )
    .then(None)
    .otherwise(
        pl.col(col).str.strip_chars()
    )
    .alias(col)
    for col in clean_string_cols
])

display(Markdown(
    f"Kolom teks yang dibersihkan: **{len(clean_string_cols)} kolom**."
))

Kolom teks yang dibersihkan: **2 kolom**.

In [64]:
STAGE_1_PATH = (
    PROCESSED_DATA_DIR /
    "momtsim_stage1_clean.parquet"
)

clean_lf.sink_parquet(
    str(STAGE_1_PATH),
    compression="zstd"
)

display(Markdown(
    f"""
### Data Cleaning Tahap Awal Selesai

File tersimpan di:

`{STAGE_1_PATH}`
"""
))

clean_lf = pl.scan_parquet(
    STAGE_1_PATH
)


### Data Cleaning Tahap Awal Selesai

File tersimpan di:

`/content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_stage1_clean.parquet`


In [65]:
import polars as pl
from IPython.display import display, Markdown

# Kolom utama berdasarkan dataset kamu
TIME_COL = "step"
TARGET_COL = "is_fraud"

# Pastikan kolom wajib tersedia
required_columns = [
    "step",
    "transaction_type",
    "amount",
    "initiator",
    "old_bal_initiator",
    "new_bal_initiator",
    "recipient",
    "old_bal_recipient",
    "new_bal_recipient",
    "is_fraud",
]

available_columns = clean_lf.collect_schema().names()

missing_columns = [
    col for col in required_columns
    if col not in available_columns
]

if missing_columns:
    raise ValueError(
        f"Kolom tidak ditemukan: {missing_columns}"
    )

print("Semua kolom wajib tersedia.")

Semua kolom wajib tersedia.


In [66]:
# ============================================================
# FEATURE ENGINEERING AWAL
# ============================================================

feature_lf = (
    clean_lf
    .with_columns([
        # Tipe data dibuat konsisten
        pl.col("step").cast(pl.Int64),
        pl.col("amount").cast(pl.Float64),
        pl.col("initiator").cast(pl.String),
        pl.col("recipient").cast(pl.String),
        pl.col("is_fraud").cast(pl.Int8).alias("target"),

        pl.col("transaction_type")
        .cast(pl.String)
        .str.strip_chars()
        .str.to_uppercase(),
    ])
    .filter(
        pl.col("step").is_not_null() &
        pl.col("amount").is_not_null() &
        pl.col("target").is_not_null()
    )
    .with_columns([
        # Mengurangi efek nominal yang sangat ekstrem
        pl.col("amount")
        .clip(lower_bound=0)
        .log1p()
        .alias("amount_log1p"),

        # Perubahan saldo aktual
        (
            pl.col("new_bal_initiator") -
            pl.col("old_bal_initiator")
        ).alias("initiator_balance_change"),

        (
            pl.col("new_bal_recipient") -
            pl.col("old_bal_recipient")
        ).alias("recipient_balance_change"),

        # Rasio nominal terhadap saldo awal
        (
            pl.col("amount") /
            (pl.col("old_bal_initiator").abs() + 1)
        ).alias("amount_to_initiator_balance_ratio"),

        (
            pl.col("amount") /
            (pl.col("old_bal_recipient").abs() + 1)
        ).alias("amount_to_recipient_balance_ratio"),

        # Seberapa besar saldo pengirim berkurang
        (
            (
                pl.col("old_bal_initiator") -
                pl.col("new_bal_initiator")
            )
            / (pl.col("old_bal_initiator").abs() + 1)
        ).alias("initiator_balance_drop_ratio"),

        # Penanda kondisi saldo tertentu
        (
            pl.col("old_bal_initiator") <= 0
        ).cast(pl.Int8).alias("initiator_old_balance_non_positive"),

        (
            pl.col("new_bal_initiator") < 0
        ).cast(pl.Int8).alias("initiator_new_balance_negative"),

        (
            pl.col("old_bal_recipient") == 0
        ).cast(pl.Int8).alias("recipient_old_balance_zero"),
    ])
)

In [67]:
feature_lf = feature_lf.with_columns([
    # Saldo pengirim yang secara sederhana diharapkan
    (
        pl.col("old_bal_initiator") -
        pl.col("amount")
    ).alias("expected_new_bal_initiator"),

    # Saldo penerima yang secara sederhana diharapkan
    (
        pl.col("old_bal_recipient") +
        pl.col("amount")
    ).alias("expected_new_bal_recipient"),
])

feature_lf = feature_lf.with_columns([
    # Selisih antara saldo aktual dan saldo yang diharapkan
    (
        pl.col("new_bal_initiator") -
        pl.col("expected_new_bal_initiator")
    ).abs().alias("initiator_balance_error"),

    (
        pl.col("new_bal_recipient") -
        pl.col("expected_new_bal_recipient")
    ).abs().alias("recipient_balance_error"),
])

In [68]:
import numpy as np
import polars as pl

# ============================================================
# FITUR WAKTU BERDASARKAN STEP
# 1 step MoMTSim = 1 jam
# ============================================================

feature_lf = feature_lf.with_columns([
    # Jam relatif dalam siklus 24 jam.
    # Ini belum tentu menunjukkan jam kalender sebenarnya
    # karena titik awal simulasi belum dikonfirmasi.
    (
        pl.col("step") % 24
    )
    .cast(pl.Int8)
    .alias("hour_of_day"),

    # Hari relatif sejak simulasi dimulai.
    (
        pl.col("step") // 24
    )
    .cast(pl.Int16)
    .alias("day_index"),
])

feature_lf = feature_lf.with_columns([
    # Fitur eksperimental.
    # Tidak akan dimasukkan ke global model V3 dahulu.
    (
        (pl.col("hour_of_day") >= 22)
        |
        (pl.col("hour_of_day") < 5)
    )
    .cast(pl.Int8)
    .alias("is_night"),

    # Representasi siklik.
    # Jam 23 dibuat dekat dengan jam 0.
    (
        2
        * np.pi
        * pl.col("hour_of_day")
        / 24
    )
    .sin()
    .alias("hour_sin"),

    (
        2
        * np.pi
        * pl.col("hour_of_day")
        / 24
    )
    .cos()
    .alias("hour_cos"),
])

# ============================================================
# PEMERIKSAAN STEP DATASET
# ============================================================

step_summary = (
    feature_lf
    .select([
        pl.col("step")
        .min()
        .alias("minimum_step"),

        pl.col("step")
        .max()
        .alias("maximum_step"),

        pl.col("step")
        .n_unique()
        .alias("unique_steps"),

        pl.col("hour_of_day")
        .min()
        .alias("minimum_relative_hour"),

        pl.col("hour_of_day")
        .max()
        .alias("maximum_relative_hour"),

        pl.col("day_index")
        .max()
        .alias("maximum_day_index"),
    ])
    .collect(
        engine="streaming"
    )
)

display(step_summary)

minimum_step,maximum_step,unique_steps,minimum_relative_hour,maximum_relative_hour,maximum_day_index
i64,i64,u32,i8,i8,i16
0,143,144,0,23,5


In [69]:
import gc
import json
import joblib
import numpy as np
import pandas as pd
import polars as pl

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
)

from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation,
)

SEED = 42

EXPERIMENT_VERSION = (
    "candidate_v3_transfer_global"
)

DATASET_VERSION = "momtsim_v1"

# ============================================================
# TARGET GLOBAL MODEL V3
# ============================================================

# Target ini khusus global supervised model.
# Belum merupakan target final aplikasi Favera.

TARGET_PR_AUC = 0.50
TARGET_PRECISION = 0.50
TARGET_RECALL = 0.70
TARGET_F1 = 0.58
TARGET_FPR = 0.20

# ============================================================
# TARGET FINAL APLIKASI
# Digunakan nanti setelah global score,
# personal anomaly score, dan rule score digabungkan.
# ============================================================

APPLICATION_TARGETS = {
    "pr_auc": 0.60,
    "precision": 0.65,
    "recall": 0.80,
    "f1": 0.70,
    "false_positive_rate": 0.10,
    "false_alerts_per_user_week": 2.0,
}

In [70]:
def build_operating_point_table(
    y_true,
    probabilities,
):
    """
    Membuat seluruh operating point berdasarkan kemungkinan threshold.
    """
    y_true = np.asarray(
        y_true,
        dtype=np.int8,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    if len(y_true) != len(probabilities):
        raise ValueError(
            "Jumlah target dan probabilitas tidak sama."
        )

    # Skor tertinggi diletakkan lebih dahulu.
    order = np.argsort(
        -probabilities,
        kind="mergesort",
    )

    sorted_target = y_true[order]
    sorted_probability = probabilities[order]

    true_positive = np.cumsum(
        sorted_target
    )

    false_positive = np.cumsum(
        1 - sorted_target
    )

    total_positive = int(
        sorted_target.sum()
    )

    total_negative = (
        len(sorted_target)
        - total_positive
    )

    false_negative = (
        total_positive
        - true_positive
    )

    true_negative = (
        total_negative
        - false_positive
    )

    precision = (
        true_positive /
        np.maximum(
            true_positive + false_positive,
            1,
        )
    )

    recall = (
        true_positive /
        max(total_positive, 1)
    )

    false_positive_rate = (
        false_positive /
        max(total_negative, 1)
    )

    f1 = (
        2 * precision * recall /
        np.maximum(
            precision + recall,
            1e-12,
        )
    )

    # Ambil posisi terakhir untuk probabilitas yang sama.
    distinct_indices = np.r_[
        np.flatnonzero(
            sorted_probability[:-1]
            != sorted_probability[1:]
        ),
        len(sorted_probability) - 1,
    ]

    return pd.DataFrame({
        "threshold": (
            sorted_probability[
                distinct_indices
            ]
        ),
        "precision": (
            precision[
                distinct_indices
            ]
        ),
        "recall": (
            recall[
                distinct_indices
            ]
        ),
        "f1": (
            f1[
                distinct_indices
            ]
        ),
        "false_positive_rate": (
            false_positive_rate[
                distinct_indices
            ]
        ),
        "true_positive": (
            true_positive[
                distinct_indices
            ]
        ),
        "false_positive": (
            false_positive[
                distinct_indices
            ]
        ),
        "false_negative": (
            false_negative[
                distinct_indices
            ]
        ),
        "true_negative": (
            true_negative[
                distinct_indices
            ]
        ),
    })

In [71]:
def evaluate_model_target(
    model_name,
    y_true,
    probabilities,
):
    """
    Memilih threshold yang memenuhi target produk.

    Jika belum ada threshold yang memenuhi seluruh target,
    dipilih titik dengan precision terbaik sambil menjaga recall >= 80%.
    """
    operating_points = (
        build_operating_point_table(
            y_true,
            probabilities,
        )
    )

    pr_auc = average_precision_score(
        y_true,
        probabilities,
    )

    feasible_points = operating_points[
        (
            operating_points["precision"]
            >= TARGET_PRECISION
        )
        &
        (
            operating_points["recall"]
            >= TARGET_RECALL
        )
        &
        (
            operating_points[
                "false_positive_rate"
            ]
            <= TARGET_FPR
        )
    ]

    if not feasible_points.empty:
        selected = (
            feasible_points
            .sort_values(
                [
                    "f1",
                    "precision",
                    "recall",
                ],
                ascending=False,
            )
            .iloc[0]
        )

        status = "TARGET ACHIEVED"

    else:
        recall_guardrail = operating_points[
            operating_points["recall"]
            >= TARGET_RECALL
        ]

        if recall_guardrail.empty:
            selected = (
                operating_points
                .sort_values(
                    "recall",
                    ascending=False,
                )
                .iloc[0]
            )
        else:
            selected = (
                recall_guardrail
                .sort_values(
                    [
                        "precision",
                        "false_positive_rate",
                        "f1",
                    ],
                    ascending=[
                        False,
                        True,
                        False,
                    ],
                )
                .iloc[0]
            )

        status = "TARGET NOT ACHIEVED"

    summary = {
        "model": model_name,
        "status": status,
        "pr_auc": float(pr_auc),
        "threshold": float(
            selected["threshold"]
        ),
        "precision": float(
            selected["precision"]
        ),
        "recall": float(
            selected["recall"]
        ),
        "f1": float(
            selected["f1"]
        ),
        "false_positive_rate": float(
            selected["false_positive_rate"]
        ),
        "true_positive": int(
            selected["true_positive"]
        ),
        "false_positive": int(
            selected["false_positive"]
        ),
        "false_negative": int(
            selected["false_negative"]
        ),
        "true_negative": int(
            selected["true_negative"]
        ),
    }

    return (
        pd.DataFrame([summary]),
        operating_points,
    )

# Histori Pengirim

In [72]:
required_history_columns = [
    "step",
    "initiator",
    "recipient",
    "amount",
]

available_feature_columns = (
    feature_lf
    .collect_schema()
    .names()
)

missing_history_columns = [
    column
    for column in required_history_columns
    if column not in available_feature_columns
]

if missing_history_columns:
    raise ValueError(
        "Kolom histori tidak ditemukan: "
        f"{missing_history_columns}"
    )

# Materialisasi hanya empat kolom yang diperlukan.
# Ini mencegah seluruh feature_lf masuk RAM sekaligus.
base_history_df = (
    feature_lf
    .select(
        required_history_columns
    )
    .collect(
        engine="streaming"
    )
)

print(
    "History source shape:",
    base_history_df.shape,
)

initiator_step_df = (
    base_history_df
    .group_by([
        "initiator",
        "step",
    ])
    .agg([
        pl.len()
        .alias(
            "initiator_tx_count_step"
        ),

        pl.col("amount")
        .sum()
        .alias(
            "initiator_amount_sum_step"
        ),

        pl.col("amount")
        .pow(2)
        .sum()
        .alias(
            "initiator_amount_sq_sum_step"
        ),

        pl.col("amount")
        .max()
        .alias(
            "initiator_amount_max_step"
        ),
    ])
    .sort([
        "initiator",
        "step",
    ])
)

print(
    "Initiator-step rows:",
    initiator_step_df.height,
)

History source shape: (1720181, 4)
Initiator-step rows: 558223


In [73]:
initiator_history_df = (
    initiator_step_df
    .with_columns([
        (
            pl.col(
                "initiator_tx_count_step"
            )
            .cum_sum()
            .over(
                "initiator",
                order_by="step",
            )
            -
            pl.col(
                "initiator_tx_count_step"
            )
        )
        .alias(
            "initiator_tx_count_prior"
        ),

        (
            pl.col(
                "initiator_amount_sum_step"
            )
            .cum_sum()
            .over(
                "initiator",
                order_by="step",
            )
            -
            pl.col(
                "initiator_amount_sum_step"
            )
        )
        .alias(
            "initiator_amount_sum_prior"
        ),

        (
            pl.col(
                "initiator_amount_sq_sum_step"
            )
            .cum_sum()
            .over(
                "initiator",
                order_by="step",
            )
            -
            pl.col(
                "initiator_amount_sq_sum_step"
            )
        )
        .alias(
            "initiator_amount_sq_sum_prior"
        ),

        pl.col(
            "initiator_amount_max_step"
        )
        .cum_max()
        .shift(1)
        .over(
            "initiator",
            order_by="step",
        )
        .alias(
            "initiator_amount_max_prior"
        ),

        pl.col("step")
        .shift(1)
        .over(
            "initiator",
            order_by="step",
        )
        .alias(
            "initiator_previous_step"
        ),
    ])
)

In [74]:
initiator_6h_df = (
    initiator_step_df
    .select([
        "initiator",
        "step",
        "initiator_tx_count_step",
        "initiator_amount_sum_step",
    ])
    .sort([
        "initiator",
        "step",
    ])
    .rolling(
        index_column="step",
        period="6i",
        closed="left",
        group_by="initiator",
    )
    .agg([
        pl.col(
            "initiator_tx_count_step"
        )
        .sum()
        .alias(
            "initiator_tx_count_6h"
        ),

        pl.col(
            "initiator_amount_sum_step"
        )
        .sum()
        .alias(
            "initiator_amount_sum_6h"
        ),
    ])
)

In [75]:
initiator_24h_df = (
    initiator_step_df
    .select([
        "initiator",
        "step",
        "initiator_tx_count_step",
        "initiator_amount_sum_step",
    ])
    .sort([
        "initiator",
        "step",
    ])
    .rolling(
        index_column="step",
        period="24i",
        closed="left",
        group_by="initiator",
    )
    .agg([
        pl.col(
            "initiator_tx_count_step"
        )
        .sum()
        .alias(
            "initiator_tx_count_24h"
        ),

        pl.col(
            "initiator_amount_sum_step"
        )
        .sum()
        .alias(
            "initiator_amount_sum_24h"
        ),
    ])
)

In [76]:
initiator_history_df = (
    initiator_history_df
    .join(
        initiator_6h_df,
        on=[
            "initiator",
            "step",
        ],
        how="left",
    )
    .join(
        initiator_24h_df,
        on=[
            "initiator",
            "step",
        ],
        how="left",
    )
)

# Histori Penerima

In [77]:
recipient_step_df = (
    base_history_df
    .group_by([
        "recipient",
        "step",
    ])
    .agg([
        pl.len()
        .alias(
            "recipient_tx_count_step"
        ),
    ])
    .sort([
        "recipient",
        "step",
    ])
)

In [78]:
recipient_history_df = (
    recipient_step_df
    .with_columns([
        (
            pl.col(
                "recipient_tx_count_step"
            )
            .cum_sum()
            .over(
                "recipient",
                order_by="step",
            )
            -
            pl.col(
                "recipient_tx_count_step"
            )
        )
        .alias(
            "recipient_tx_count_prior"
        ),
    ])
    .select([
        "recipient",
        "step",
        "recipient_tx_count_prior",
    ])
)

# Histori Pasangan pengirim dan Penerima

In [79]:
pair_step_df = (
    base_history_df
    .group_by([
        "initiator",
        "recipient",
        "step",
    ])
    .agg([
        pl.len()
        .alias(
            "pair_tx_count_step"
        ),
    ])
    .sort([
        "initiator",
        "recipient",
        "step",
    ])
)

pair_history_df = (
    pair_step_df
    .with_columns([
        (
            pl.col(
                "pair_tx_count_step"
            )
            .cum_sum()
            .over(
                [
                    "initiator",
                    "recipient",
                ],
                order_by="step",
            )
            -
            pl.col(
                "pair_tx_count_step"
            )
        )
        .alias(
            "pair_tx_count_prior"
        ),

        pl.col("step")
        .shift(1)
        .over(
            [
                "initiator",
                "recipient",
            ],
            order_by="step",
        )
        .alias(
            "pair_previous_step"
        ),
    ])
    .select([
        "initiator",
        "recipient",
        "step",
        "pair_tx_count_prior",
        "pair_previous_step",
    ])
)

In [80]:
behavioral_feature_lf = (
    feature_lf
    .join(
        initiator_history_df.lazy(),
        on=[
            "initiator",
            "step",
        ],
        how="left",
    )
    .join(
        recipient_history_df.lazy(),
        on=[
            "recipient",
            "step",
        ],
        how="left",
    )
    .join(
        pair_history_df.lazy(),
        on=[
            "initiator",
            "recipient",
            "step",
        ],
        how="left",
    )
)

# Pengisian Null Value

In [81]:
history_zero_columns = [
    "initiator_tx_count_prior",
    "initiator_amount_sum_prior",
    "initiator_amount_sq_sum_prior",
    "initiator_amount_max_prior",
    "initiator_tx_count_6h",
    "initiator_amount_sum_6h",
    "initiator_tx_count_24h",
    "initiator_amount_sum_24h",
    "recipient_tx_count_prior",
    "pair_tx_count_prior",
]

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        pl.col(column)
        .fill_null(0)
        .alias(column)
        for column in history_zero_columns
    ])
)

# Fitur Behavioral V2

In [82]:
behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        (
            pl.col(
                "initiator_tx_count_prior"
            ) > 0
        )
        .cast(pl.Int8)
        .alias(
            "has_initiator_history"
        ),

        (
            pl.col(
                "pair_tx_count_prior"
            ) == 0
        )
        .cast(pl.Int8)
        .alias(
            "is_new_recipient"
        ),

        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 0
        )
        .then(
            pl.col(
                "initiator_amount_sum_prior"
            )
            /
            pl.col(
                "initiator_tx_count_prior"
            )
        )
        .otherwise(0)
        .alias(
            "initiator_amount_mean_prior"
        ),
    ])
)

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 1
        )
        .then(
            (
                pl.col(
                    "initiator_amount_sq_sum_prior"
                )
                /
                pl.col(
                    "initiator_tx_count_prior"
                )
            )
            -
            pl.col(
                "initiator_amount_mean_prior"
            ).pow(2)
        )
        .otherwise(0)
        .clip(lower_bound=0)
        .alias(
            "initiator_amount_variance_prior"
        ),
    ])
    .with_columns([
        pl.col(
            "initiator_amount_variance_prior"
        )
        .sqrt()
        .alias(
            "initiator_amount_std_prior"
        ),
    ])
)

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 0
        )
        .then(
            pl.col("amount")
            /
            (
                pl.col(
                    "initiator_amount_mean_prior"
                ).abs()
                + 1
            )
        )
        .otherwise(0)
        .alias(
            "amount_vs_initiator_mean"
        ),

        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 1
        )
        .then(
            (
                pl.col("amount")
                -
                pl.col(
                    "initiator_amount_mean_prior"
                )
            )
            /
            (
                pl.col(
                    "initiator_amount_std_prior"
                )
                + 1
            )
        )
        .otherwise(0)
        .alias(
            "amount_zscore_initiator"
        ),

        pl.when(
            pl.col(
                "initiator_amount_max_prior"
            ) > 0
        )
        .then(
            pl.col("amount")
            /
            (
                pl.col(
                    "initiator_amount_max_prior"
                )
                + 1
            )
        )
        .otherwise(0)
        .alias(
            "amount_vs_initiator_max"
        ),

        pl.when(
            pl.col(
                "initiator_previous_step"
            ).is_not_null()
        )
        .then(
            pl.col("step")
            -
            pl.col(
                "initiator_previous_step"
            )
        )
        .otherwise(-1)
        .alias(
            "steps_since_initiator_transaction"
        ),

        pl.when(
            pl.col(
                "pair_previous_step"
            ).is_not_null()
        )
        .then(
            pl.col("step")
            -
            pl.col(
                "pair_previous_step"
            )
        )
        .otherwise(-1)
        .alias(
            "steps_since_pair_transaction"
        ),
    ])
)

# Fitur V3

In [83]:
# ============================================================
# 03.5 CANDIDATE V3 DERIVED BEHAVIORAL FEATURES
# ============================================================

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        (
            pl.col("amount_log1p")
            -
            pl.col("initiator_amount_mean_prior")
            .clip(lower_bound=0)
            .log1p()
        )
        .alias("amount_log_ratio_to_mean"),

        (
            pl.col("initiator_tx_count_6h")
            /
            (
                pl.col("initiator_tx_count_24h") / 4
                + 1
            )
        )
        .alias("tx_velocity_6h_vs_24h"),

        pl.when(
            pl.col("initiator_tx_count_6h") > 0
        )
        .then(
            pl.col("amount")
            /
            (
                pl.col("initiator_amount_sum_6h")
                /
                pl.col("initiator_tx_count_6h")
                + 1
            )
        )
        .otherwise(0)
        .alias("amount_vs_recent_mean_6h"),

        pl.when(
            pl.col("initiator_tx_count_24h") > 0
        )
        .then(
            pl.col("amount")
            /
            (
                pl.col("initiator_amount_sum_24h")
                /
                pl.col("initiator_tx_count_24h")
                + 1
            )
        )
        .otherwise(0)
        .alias("amount_vs_recent_mean_24h"),

        pl.col("recipient_tx_count_prior")
        .cast(pl.Float64)
        .log1p()
        .alias("recipient_history_log1p"),

        pl.col("pair_tx_count_prior")
        .cast(pl.Float64)
        .log1p()
        .alias("pair_history_log1p"),

        (
            (
                pl.col("steps_since_initiator_transaction")
                >= 0
            )
            &
            (
                pl.col("steps_since_initiator_transaction")
                <= 1
            )
        )
        .cast(pl.Int8)
        .alias("is_rapid_initiator_repeat"),

        (
            (
                pl.col("steps_since_pair_transaction")
                >= 0
            )
            &
            (
                pl.col("steps_since_pair_transaction")
                <= 1
            )
        )
        .cast(pl.Int8)
        .alias("is_rapid_pair_repeat"),

        (
            pl.col("new_bal_initiator")
            /
            (
                pl.col("old_bal_initiator").abs()
                + 1
            )
        )
        .alias("remaining_balance_ratio"),

        (
            pl.col("initiator_tx_count_prior") < 5
        )
        .cast(pl.Int8)
        .alias("is_cold_start"),
    ])
)

In [84]:
behavioral_preview = (
    behavioral_feature_lf
    .select([
        "step",
        "initiator",
        "recipient",
        "amount",
        "initiator_tx_count_prior",
        "initiator_amount_mean_prior",
        "amount_vs_initiator_mean",
        "initiator_tx_count_6h",
        "initiator_tx_count_24h",
        "recipient_tx_count_prior",
        "pair_tx_count_prior",
        "is_new_recipient",
        "target",
    ])
    .head(10)
    .collect()
)

display(behavioral_preview)

step,initiator,recipient,amount,initiator_tx_count_prior,initiator_amount_mean_prior,amount_vs_initiator_mean,initiator_tx_count_6h,initiator_tx_count_24h,recipient_tx_count_prior,pair_tx_count_prior,is_new_recipient,target
i64,str,str,f64,u32,f64,f64,u32,u32,u32,u32,i8,i8
0,"""4537027967639631""","""4875702729424478""",19824.96,0,0.0,0.0,0,0,0,0,1,1
0,"""4296267625767470""","""25-0000401""",598.97,0,0.0,0.0,0,0,0,0,1,0
0,"""4178224023847746""","""13-0001587""",545.85,0,0.0,0.0,0,0,0,0,1,0
0,"""4178224023847746""","""4096920916696293""",19847.01,0,0.0,0.0,0,0,0,0,1,1
0,"""4779013371563747""","""75-0003564""",546.89,0,0.0,0.0,0,0,0,0,1,0
0,"""4779013371563747""","""4593247015015219""",16973.75,0,0.0,0.0,0,0,0,0,1,1
0,"""4479448076398077""","""11-0003866""",659.02,0,0.0,0.0,0,0,0,0,1,0
0,"""4479448076398077""","""4649550475321678""",21103.99,0,0.0,0.0,0,0,0,0,1,1
0,"""4587942659105635""","""64-0008405""",648.89,0,0.0,0.0,0,0,0,0,1,0


In [85]:
behavioral_row_count = (
    behavioral_feature_lf
    .select(
        pl.len()
    )
    .collect(
        engine="streaming"
    )
    .item()
)

print(
    "Rows before behavioral features:",
    row_count,
)

print(
    "Rows after behavioral features:",
    behavioral_row_count,
)

if behavioral_row_count != row_count:
    raise ValueError(
        "Jumlah baris berubah setelah join behavioral features."
    )

Rows before behavioral features: 1720181
Rows after behavioral features: 1720181


In [86]:
# ============================================================
# AUDIT LABEL BERDASARKAN TIPE TRANSAKSI
# ============================================================

fraud_by_transaction_type_v3 = (
    behavioral_feature_lf
    .group_by("transaction_type")
    .agg([
        pl.len()
        .alias("transaction_count"),

        pl.col("target")
        .sum()
        .alias("fraud_count"),

        pl.col("target")
        .mean()
        .alias("fraud_rate"),
    ])
    .with_columns([
        (
            pl.col("transaction_count")
            - pl.col("fraud_count")
        )
        .alias("normal_count")
    ])
    .sort(
        "fraud_count",
        descending=True,
    )
    .collect(
        engine="streaming"
    )
)

display(
    fraud_by_transaction_type_v3
)

non_transfer_fraud_count_v3 = (
    behavioral_feature_lf
    .filter(
        (pl.col("transaction_type") != "TRANSFER")
        &
        (pl.col("target") == 1)
    )
    .select(
        pl.len()
    )
    .collect(
        engine="streaming"
    )
    .item()
)

print(
    "Fraud di luar TRANSFER:",
    non_transfer_fraud_count_v3,
)

if non_transfer_fraud_count_v3 > 0:
    raise ValueError(
        "Ditemukan fraud di luar TRANSFER. "
        "Filter global model perlu ditinjau ulang."
    )

transaction_type,transaction_count,fraud_count,fraud_rate,normal_count
str,u32,i64,f64,i64
"""TRANSFER""",569328,175518,0.30829,393810
"""DEPOSIT""",384431,0,0.0,384431
"""PAYMENT""",667245,0,0.0,667245
"""DEBIT""",5392,0,0.0,5392
"""WITHDRAWAL""",93785,0,0.0,93785


Fraud di luar TRANSFER: 0


In [87]:
# ============================================================
# TEMPORAL SPLIT CANDIDATE V3
# Train 70%
# Validation 10%
# Calibration 10%
# Test 10%
# ============================================================

unique_steps_v3 = (
    behavioral_feature_lf
    .select("step")
    .unique()
    .sort("step")
    .collect(
        engine="streaming"
    )
    .get_column("step")
)

number_of_steps_v3 = len(
    unique_steps_v3
)

if number_of_steps_v3 < 40:
    raise ValueError(
        "Jumlah step terlalu sedikit "
        "untuk empat temporal split."
    )

train_step_index_v3 = (
    int(
        number_of_steps_v3 * 0.70
    )
    - 1
)

validation_step_index_v3 = (
    int(
        number_of_steps_v3 * 0.80
    )
    - 1
)

calibration_step_index_v3 = (
    int(
        number_of_steps_v3 * 0.90
    )
    - 1
)

train_end_step_v3 = (
    unique_steps_v3[
        train_step_index_v3
    ]
)

validation_end_step_v3 = (
    unique_steps_v3[
        validation_step_index_v3
    ]
)

calibration_end_step_v3 = (
    unique_steps_v3[
        calibration_step_index_v3
    ]
)

print(
    "Jumlah step:",
    number_of_steps_v3,
)

print(
    "Train:",
    unique_steps_v3[0],
    "sampai",
    train_end_step_v3,
)

print(
    "Validation:",
    train_end_step_v3 + 1,
    "sampai",
    validation_end_step_v3,
)

print(
    "Calibration:",
    validation_end_step_v3 + 1,
    "sampai",
    calibration_end_step_v3,
)

print(
    "Test:",
    calibration_end_step_v3 + 1,
    "sampai",
    unique_steps_v3[-1],
)

Jumlah step: 144
Train: 0 sampai 99
Validation: 100 sampai 114
Calibration: 115 sampai 128
Test: 129 sampai 143


In [88]:
# ============================================================
# GLOBAL MODEL V3
# HANYA TRANSAKSI TRANSFER
# ============================================================

global_model_filter_v3 = (
    pl.col("transaction_type")
    == "TRANSFER"
)

train_v3_lf = (
    behavioral_feature_lf
    .filter(
        global_model_filter_v3
        &
        (
            pl.col("step")
            <= train_end_step_v3
        )
    )
)

validation_v3_lf = (
    behavioral_feature_lf
    .filter(
        global_model_filter_v3
        &
        (
            pl.col("step")
            > train_end_step_v3
        )
        &
        (
            pl.col("step")
            <= validation_end_step_v3
        )
    )
)

calibration_v3_lf = (
    behavioral_feature_lf
    .filter(
        global_model_filter_v3
        &
        (
            pl.col("step")
            > validation_end_step_v3
        )
        &
        (
            pl.col("step")
            <= calibration_end_step_v3
        )
    )
)

test_v3_lf = (
    behavioral_feature_lf
    .filter(
        global_model_filter_v3
        &
        (
            pl.col("step")
            > calibration_end_step_v3
        )
    )
)

In [89]:
# ============================================================
# RINGKASAN TEMPORAL SPLIT V3
# ============================================================

def summarize_split_v3(
    split_name,
    split_lf,
):
    return (
        split_lf
        .select([
            pl.lit(
                split_name
            )
            .alias("split"),

            pl.len()
            .alias("row_count"),

            pl.col("step")
            .min()
            .alias("step_start"),

            pl.col("step")
            .max()
            .alias("step_end"),

            pl.col("target")
            .sum()
            .alias("fraud_count"),

            (
                pl.len()
                - pl.col("target").sum()
            )
            .alias("normal_count"),

            pl.col("target")
            .mean()
            .alias("fraud_rate"),

            pl.col("initiator")
            .n_unique()
            .alias("unique_initiators"),

            pl.col("recipient")
            .n_unique()
            .alias("unique_recipients"),
        ])
        .collect(
            engine="streaming"
        )
    )


split_summary_v3 = pl.concat([
    summarize_split_v3(
        "train",
        train_v3_lf,
    ),
    summarize_split_v3(
        "validation",
        validation_v3_lf,
    ),
    summarize_split_v3(
        "calibration",
        calibration_v3_lf,
    ),
    summarize_split_v3(
        "test",
        test_v3_lf,
    ),
])

display(
    split_summary_v3
)

for split_row in split_summary_v3.iter_rows(
    named=True
):
    if split_row["row_count"] == 0:
        raise ValueError(
            f"Split {split_row['split']} kosong."
        )

    if split_row["fraud_count"] == 0:
        raise ValueError(
            f"Split {split_row['split']} "
            "tidak memiliki fraud."
        )

    if split_row["normal_count"] == 0:
        raise ValueError(
            f"Split {split_row['split']} "
            "tidak memiliki transaksi normal."
        )

split,row_count,step_start,step_end,fraud_count,normal_count,fraud_rate,unique_initiators,unique_recipients
str,u32,i64,i64,i64,i64,f64,u32,u32
"""train""",391716,0,99,122258,269458,0.312109,10863,9593
"""validation""",58895,100,114,18208,40687,0.30916,7629,5000
"""calibration""",55066,115,128,16915,38151,0.307177,7522,5190
"""test""",63651,129,143,18137,45514,0.284944,7946,7708


In [90]:
# ============================================================
# PATH DATASET CANDIDATE V3
# ============================================================

TRAIN_V3_PATH = (
    PROCESSED_DATA_DIR
    / "momtsim_train_features_v3.parquet"
)

VALIDATION_V3_PATH = (
    PROCESSED_DATA_DIR
    / "momtsim_validation_features_v3.parquet"
)

CALIBRATION_V3_PATH = (
    PROCESSED_DATA_DIR
    / "momtsim_calibration_features_v3.parquet"
)

TEST_V3_PATH = (
    PROCESSED_DATA_DIR
    / "momtsim_test_features_v3.parquet"
)

print(
    "Train path:",
    TRAIN_V3_PATH,
)

print(
    "Validation path:",
    VALIDATION_V3_PATH,
)

print(
    "Calibration path:",
    CALIBRATION_V3_PATH,
)

print(
    "Test path:",
    TEST_V3_PATH,
)

Train path: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_train_features_v3.parquet
Validation path: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_validation_features_v3.parquet
Calibration path: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_calibration_features_v3.parquet
Test path: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_test_features_v3.parquet


In [91]:
# ============================================================
# SIMPAN TEMPORAL SPLIT V3
# ============================================================

train_v3_lf.sink_parquet(
    str(TRAIN_V3_PATH),
    compression="zstd",
)

validation_v3_lf.sink_parquet(
    str(VALIDATION_V3_PATH),
    compression="zstd",
)

calibration_v3_lf.sink_parquet(
    str(CALIBRATION_V3_PATH),
    compression="zstd",
)

test_v3_lf.sink_parquet(
    str(TEST_V3_PATH),
    compression="zstd",
)

print(
    "Train V3 tersimpan:",
    TRAIN_V3_PATH,
)

print(
    "Validation V3 tersimpan:",
    VALIDATION_V3_PATH,
)

print(
    "Calibration V3 tersimpan:",
    CALIBRATION_V3_PATH,
)

print(
    "Test V3 tersimpan:",
    TEST_V3_PATH,
)

Train V3 tersimpan: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_train_features_v3.parquet
Validation V3 tersimpan: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_validation_features_v3.parquet
Calibration V3 tersimpan: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_calibration_features_v3.parquet
Test V3 tersimpan: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_test_features_v3.parquet


In [92]:
del base_history_df
del initiator_step_df
del initiator_6h_df
del initiator_24h_df
del recipient_step_df
del pair_step_df

gc.collect()

print("Intermediate memory dibersihkan.")

Intermediate memory dibersihkan.


In [93]:
# ============================================================
# FEATURE CONTRACT CANDIDATE V3
# ============================================================

V3_NUMERIC_FEATURES = [
    # --------------------------------------------------------
    # Nominal transaksi
    # --------------------------------------------------------
    "amount",
    "amount_log1p",

    # --------------------------------------------------------
    # Kondisi saldo pengirim
    # Fitur saldo penerima sengaja tidak digunakan
    # karena umumnya tidak tersedia dalam e-statement.
    # --------------------------------------------------------
    "old_bal_initiator",
    "new_bal_initiator",
    "initiator_balance_change",
    "amount_to_initiator_balance_ratio",
    "initiator_balance_drop_ratio",
    "initiator_old_balance_non_positive",
    "initiator_new_balance_negative",
    "initiator_balance_error",
    "remaining_balance_ratio",

    # --------------------------------------------------------
    # Waktu
    # 1 step MoMTSim = 1 jam
    # --------------------------------------------------------
    "hour_of_day",
    "is_night",
    "hour_sin",
    "hour_cos",

    # --------------------------------------------------------
    # Histori pengirim
    # --------------------------------------------------------
    "has_initiator_history",
    "initiator_tx_count_prior",
    "initiator_amount_mean_prior",
    "initiator_amount_std_prior",
    "initiator_amount_max_prior",
    "amount_vs_initiator_mean",
    "amount_zscore_initiator",
    "amount_vs_initiator_max",
    "amount_log_ratio_to_mean",

    # --------------------------------------------------------
    # Aktivitas terbaru
    # --------------------------------------------------------
    "initiator_tx_count_6h",
    "initiator_amount_sum_6h",
    "initiator_tx_count_24h",
    "initiator_amount_sum_24h",
    "tx_velocity_6h_vs_24h",
    "amount_vs_recent_mean_6h",
    "amount_vs_recent_mean_24h",
    "steps_since_initiator_transaction",
    "is_rapid_initiator_repeat",

    # --------------------------------------------------------
    # Histori penerima dan pasangan
    # --------------------------------------------------------
    "recipient_tx_count_prior",
    "recipient_history_log1p",
    "pair_tx_count_prior",
    "pair_history_log1p",
    "is_new_recipient",
    "steps_since_pair_transaction",
    "is_rapid_pair_repeat",

    # --------------------------------------------------------
    # Cold-start
    # --------------------------------------------------------
    "is_cold_start",
]

# Kolom ini tidak masuk ke model.
# Kolom dipertahankan untuk audit hasil prediksi.
V3_AUDIT_COLUMNS = [
    "step",
    "initiator",
    "recipient",
    "transaction_type",
]

V3_TARGET = "target"

print(
    "Jumlah fitur numerik V3:",
    len(V3_NUMERIC_FEATURES),
)

Jumlah fitur numerik V3: 41


In [94]:
# ============================================================
# LOADER DATASET CANDIDATE V3
# ============================================================

def load_v3_split(path):
    selected_columns = (
        V3_AUDIT_COLUMNS
        + V3_NUMERIC_FEATURES
        + [V3_TARGET]
    )

    lazy_split = pl.scan_parquet(
        str(path)
    )

    available_columns = (
        lazy_split
        .collect_schema()
        .names()
    )

    missing_columns = [
        column
        for column in selected_columns
        if column not in available_columns
    ]

    if missing_columns:
        raise ValueError(
            f"Kolom V3 tidak ditemukan pada {path}: "
            f"{missing_columns}"
        )

    numeric_expressions = [
        pl.col(column)
        .cast(
            pl.Float32,
            strict=False,
        )
        .fill_nan(0)
        .fill_null(0)
        .alias(column)
        for column in V3_NUMERIC_FEATURES
    ]

    loaded_df = (
        lazy_split
        .select(selected_columns)
        .with_columns(
            numeric_expressions
            + [
                pl.col("step")
                .cast(pl.Int64),

                pl.col("initiator")
                .cast(pl.String)
                .fill_null("UNKNOWN"),

                pl.col("recipient")
                .cast(pl.String)
                .fill_null("UNKNOWN"),

                pl.col("transaction_type")
                .cast(pl.String)
                .fill_null("UNKNOWN"),

                pl.col(V3_TARGET)
                .cast(pl.Int8)
                .alias(V3_TARGET),
            ]
        )
        .collect(
            engine="streaming"
        )
        .to_pandas()
    )

    return loaded_df

In [95]:
# ============================================================
# LOAD EMPAT TEMPORAL SPLIT V3
# ============================================================

train_v3_df = load_v3_split(
    TRAIN_V3_PATH
)

validation_v3_df = load_v3_split(
    VALIDATION_V3_PATH
)

calibration_v3_df = load_v3_split(
    CALIBRATION_V3_PATH
)

test_v3_df = load_v3_split(
    TEST_V3_PATH
)

print(
    "Train V3:",
    train_v3_df.shape,
)

print(
    "Validation V3:",
    validation_v3_df.shape,
)

print(
    "Calibration V3:",
    calibration_v3_df.shape,
)

print(
    "Test V3:",
    test_v3_df.shape,
)

Train V3: (391716, 46)
Validation V3: (58895, 46)
Calibration V3: (55066, 46)
Test V3: (63651, 46)


In [96]:
# ============================================================
# AUDIT DATASET YANG SUDAH DIMUAT
# ============================================================

loaded_split_summary_v3 = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train_v3_df),
        "fraud_count": int(
            train_v3_df[V3_TARGET].sum()
        ),
        "fraud_rate": float(
            train_v3_df[V3_TARGET].mean()
        ),
        "step_start": int(
            train_v3_df["step"].min()
        ),
        "step_end": int(
            train_v3_df["step"].max()
        ),
    },
    {
        "split": "validation",
        "rows": len(validation_v3_df),
        "fraud_count": int(
            validation_v3_df[V3_TARGET].sum()
        ),
        "fraud_rate": float(
            validation_v3_df[V3_TARGET].mean()
        ),
        "step_start": int(
            validation_v3_df["step"].min()
        ),
        "step_end": int(
            validation_v3_df["step"].max()
        ),
    },
    {
        "split": "calibration",
        "rows": len(calibration_v3_df),
        "fraud_count": int(
            calibration_v3_df[V3_TARGET].sum()
        ),
        "fraud_rate": float(
            calibration_v3_df[V3_TARGET].mean()
        ),
        "step_start": int(
            calibration_v3_df["step"].min()
        ),
        "step_end": int(
            calibration_v3_df["step"].max()
        ),
    },
    {
        "split": "test",
        "rows": len(test_v3_df),
        "fraud_count": int(
            test_v3_df[V3_TARGET].sum()
        ),
        "fraud_rate": float(
            test_v3_df[V3_TARGET].mean()
        ),
        "step_start": int(
            test_v3_df["step"].min()
        ),
        "step_end": int(
            test_v3_df["step"].max()
        ),
    },
])

display(
    loaded_split_summary_v3
)

for split_name, split_df in [
    ("train", train_v3_df),
    ("validation", validation_v3_df),
    ("calibration", calibration_v3_df),
    ("test", test_v3_df),
]:
    transaction_types = (
        split_df["transaction_type"]
        .dropna()
        .unique()
        .tolist()
    )

    print(
        split_name,
        "transaction types:",
        transaction_types,
    )

    if transaction_types != ["TRANSFER"]:
        raise ValueError(
            f"Split {split_name} masih memiliki "
            f"tipe selain TRANSFER: "
            f"{transaction_types}"
        )

,split,rows,fraud_count,fraud_rate,step_start,step_end
0,train,391716,122258,0.312109,0,99
1,validation,58895,18208,0.309160,100,114
2,calibration,55066,16915,0.307177,115,128
3,test,63651,18137,0.284944,129,143


train transaction types: ['TRANSFER']
validation transaction types: ['TRANSFER']
calibration transaction types: ['TRANSFER']
test transaction types: ['TRANSFER']


In [97]:
# ============================================================
# MEMBENTUK MATRIKS MODEL V3
# ============================================================

def build_model_matrix_v3(
    dataframe,
):
    X = (
        dataframe[
            V3_NUMERIC_FEATURES
        ]
        .copy()
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .fillna(0)
        .astype("float32")
    )

    y = (
        dataframe[V3_TARGET]
        .astype("int8")
        .copy()
    )

    audit = (
        dataframe[
            V3_AUDIT_COLUMNS
        ]
        .copy()
    )

    return X, y, audit


(
    X_train_v3,
    y_train_v3,
    train_audit_v3,
) = build_model_matrix_v3(
    train_v3_df
)

(
    X_validation_v3,
    y_validation_v3,
    validation_audit_v3,
) = build_model_matrix_v3(
    validation_v3_df
)

(
    X_calibration_v3,
    y_calibration_v3,
    calibration_audit_v3,
) = build_model_matrix_v3(
    calibration_v3_df
)

(
    X_test_v3,
    y_test_v3,
    test_audit_v3,
) = build_model_matrix_v3(
    test_v3_df
)

In [98]:
# ============================================================
# VALIDASI MATRIKS MODEL V3
# ============================================================

matrix_summary_v3 = pd.DataFrame([
    {
        "split": "train",
        "rows": X_train_v3.shape[0],
        "features": X_train_v3.shape[1],
        "fraud_rate": y_train_v3.mean(),
        "missing_values": int(
            X_train_v3
            .isna()
            .sum()
            .sum()
        ),
        "infinite_values": int(
            np.isinf(
                X_train_v3.to_numpy()
            ).sum()
        ),
    },
    {
        "split": "validation",
        "rows": X_validation_v3.shape[0],
        "features": X_validation_v3.shape[1],
        "fraud_rate": y_validation_v3.mean(),
        "missing_values": int(
            X_validation_v3
            .isna()
            .sum()
            .sum()
        ),
        "infinite_values": int(
            np.isinf(
                X_validation_v3.to_numpy()
            ).sum()
        ),
    },
    {
        "split": "calibration",
        "rows": X_calibration_v3.shape[0],
        "features": X_calibration_v3.shape[1],
        "fraud_rate": y_calibration_v3.mean(),
        "missing_values": int(
            X_calibration_v3
            .isna()
            .sum()
            .sum()
        ),
        "infinite_values": int(
            np.isinf(
                X_calibration_v3.to_numpy()
            ).sum()
        ),
    },
    {
        "split": "test",
        "rows": X_test_v3.shape[0],
        "features": X_test_v3.shape[1],
        "fraud_rate": y_test_v3.mean(),
        "missing_values": int(
            X_test_v3
            .isna()
            .sum()
            .sum()
        ),
        "infinite_values": int(
            np.isinf(
                X_test_v3.to_numpy()
            ).sum()
        ),
    },
])

display(
    matrix_summary_v3
)

assert list(X_train_v3.columns) == V3_NUMERIC_FEATURES
assert list(X_validation_v3.columns) == V3_NUMERIC_FEATURES
assert list(X_calibration_v3.columns) == V3_NUMERIC_FEATURES
assert list(X_test_v3.columns) == V3_NUMERIC_FEATURES

assert (
    matrix_summary_v3["missing_values"]
    == 0
).all()

assert (
    matrix_summary_v3["infinite_values"]
    == 0
).all()

print(
    "Matriks Candidate V3 siap untuk training."
)

,split,rows,features,fraud_rate,missing_values,infinite_values
0,train,391716,41,0.312109,0,0
1,validation,58895,41,0.309160,0,0
2,calibration,55066,41,0.307177,0,0
3,test,63651,41,0.284944,0,0


Matriks Candidate V3 siap untuk training.


In [99]:
# ============================================================
# THRESHOLD CALIBRATION DAN FINAL EVALUATION V3
# ============================================================

def select_threshold_v3(
    model_name,
    y_true,
    probabilities,
):
    """
    Memilih threshold hanya menggunakan calibration set.
    """
    operating_points = build_operating_point_table(
        y_true=y_true,
        probabilities=probabilities,
    )

    pr_auc = average_precision_score(
        y_true,
        probabilities,
    )

    # Kandidat yang memenuhi seluruh target global model
    feasible_points = operating_points[
        (
            operating_points["precision"]
            >= TARGET_PRECISION
        )
        &
        (
            operating_points["recall"]
            >= TARGET_RECALL
        )
        &
        (
            operating_points["f1"]
            >= TARGET_F1
        )
        &
        (
            operating_points[
                "false_positive_rate"
            ]
            <= TARGET_FPR
        )
    ].copy()

    if not feasible_points.empty:
        selected = (
            feasible_points
            .sort_values(
                [
                    "f1",
                    "precision",
                    "false_positive_rate",
                ],
                ascending=[
                    False,
                    False,
                    True,
                ],
            )
            .iloc[0]
        )

        selection_strategy = (
            "ALL GLOBAL TARGETS ACHIEVED"
        )

    else:
        # Jika target penuh belum tercapai,
        # prioritaskan recall dan kendalikan FPR.
        guardrail_points = operating_points[
            (
                operating_points["recall"]
                >= TARGET_RECALL
            )
            &
            (
                operating_points[
                    "false_positive_rate"
                ]
                <= max(TARGET_FPR, 0.25)
            )
        ].copy()

        if not guardrail_points.empty:
            selected = (
                guardrail_points
                .sort_values(
                    [
                        "precision",
                        "f1",
                        "false_positive_rate",
                    ],
                    ascending=[
                        False,
                        False,
                        True,
                    ],
                )
                .iloc[0]
            )

            selection_strategy = (
                "RECALL AND FPR GUARDRAIL"
            )

        else:
            recall_points = operating_points[
                operating_points["recall"]
                >= TARGET_RECALL
            ].copy()

            if not recall_points.empty:
                selected = (
                    recall_points
                    .sort_values(
                        [
                            "f1",
                            "precision",
                            "false_positive_rate",
                        ],
                        ascending=[
                            False,
                            False,
                            True,
                        ],
                    )
                    .iloc[0]
                )

                selection_strategy = (
                    "RECALL GUARDRAIL"
                )

            else:
                selected = (
                    operating_points
                    .sort_values(
                        [
                            "f1",
                            "precision",
                            "recall",
                        ],
                        ascending=[
                            False,
                            False,
                            False,
                        ],
                    )
                    .iloc[0]
                )

                selection_strategy = (
                    "BEST AVAILABLE F1"
                )

    target_achieved = (
        pr_auc >= TARGET_PR_AUC
        and selected["precision"]
        >= TARGET_PRECISION
        and selected["recall"]
        >= TARGET_RECALL
        and selected["f1"]
        >= TARGET_F1
        and selected[
            "false_positive_rate"
        ]
        <= TARGET_FPR
    )

    summary = pd.DataFrame([{
        "model": model_name,
        "status": (
            "TARGET ACHIEVED"
            if target_achieved
            else "TARGET NOT ACHIEVED"
        ),
        "selection_strategy": (
            selection_strategy
        ),
        "pr_auc": float(pr_auc),
        "threshold": float(
            selected["threshold"]
        ),
        "precision": float(
            selected["precision"]
        ),
        "recall": float(
            selected["recall"]
        ),
        "f1": float(
            selected["f1"]
        ),
        "false_positive_rate": float(
            selected[
                "false_positive_rate"
            ]
        ),
        "true_positive": int(
            selected["true_positive"]
        ),
        "false_positive": int(
            selected["false_positive"]
        ),
        "false_negative": int(
            selected["false_negative"]
        ),
        "true_negative": int(
            selected["true_negative"]
        ),
    }])

    return summary, operating_points


def evaluate_fixed_threshold_v3(
    model_name,
    y_true,
    probabilities,
    threshold,
):
    """
    Mengevaluasi validation atau test menggunakan
    threshold yang sudah dipilih dari calibration set.
    """
    y_true = np.asarray(
        y_true,
        dtype=np.int8,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    predictions = (
        probabilities >= threshold
    ).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    precision = (
        tp / max(tp + fp, 1)
    )

    recall = (
        tp / max(tp + fn, 1)
    )

    f1 = (
        2 * precision * recall
        / max(
            precision + recall,
            1e-12,
        )
    )

    false_positive_rate = (
        fp / max(fp + tn, 1)
    )

    fraud_rate = float(
        y_true.mean()
    )

    pr_auc = average_precision_score(
        y_true,
        probabilities,
    )

    target_achieved = (
        pr_auc >= TARGET_PR_AUC
        and precision >= TARGET_PRECISION
        and recall >= TARGET_RECALL
        and f1 >= TARGET_F1
        and false_positive_rate
        <= TARGET_FPR
    )

    return pd.DataFrame([{
        "model": model_name,
        "status": (
            "TARGET ACHIEVED"
            if target_achieved
            else "TARGET NOT ACHIEVED"
        ),
        "rows": len(y_true),
        "fraud_rate": fraud_rate,
        "pr_auc": float(pr_auc),
        "threshold": float(threshold),
        "precision": float(precision),
        "precision_lift": float(
            precision
            / max(fraud_rate, 1e-12)
        ),
        "recall": float(recall),
        "f1": float(f1),
        "false_positive_rate": float(
            false_positive_rate
        ),
        "alert_rate": float(
            predictions.mean()
        ),
        "true_positive": int(tp),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_negative": int(tn),
    }])

In [100]:
# ============================================================
# TRAIN LIGHTGBM CANDIDATE V3
# ============================================================

lightgbm_v3_model = LGBMClassifier(
    objective="binary",

    # Banyak estimator dengan learning rate kecil
    # agar proses pembelajaran lebih stabil.
    n_estimators=2500,
    learning_rate=0.03,

    # Kompleksitas pohon dikendalikan
    # agar tidak mudah menghafal pola sintetis.
    num_leaves=31,
    max_depth=8,
    min_child_samples=200,
    min_split_gain=0.01,

    # Sampling fitur dan baris.
    subsample=0.80,
    subsample_freq=1,
    colsample_bytree=0.80,

    # Regularisasi.
    reg_alpha=0.50,
    reg_lambda=2.00,

    max_bin=255,
    class_weight=None,

    random_state=SEED,
    n_jobs=-1,
    force_row_wise=True,
    verbosity=-1,
)

lightgbm_v3_model.fit(
    X_train_v3,
    y_train_v3,

    # Validation hanya digunakan untuk early stopping.
    eval_set=[
        (
            X_validation_v3,
            y_validation_v3,
        )
    ],

    eval_metric="average_precision",

    callbacks=[
        early_stopping(
            stopping_rounds=120,
            first_metric_only=True,
            verbose=True,
        ),

        log_evaluation(
            period=50,
        ),
    ],
)

print(
    "Best iteration:",
    lightgbm_v3_model.best_iteration_,
)

Training until validation scores don't improve for 120 rounds
[50]	valid_0's average_precision: 0.356516	valid_0's binary_logloss: 0.60781
[100]	valid_0's average_precision: 0.356324	valid_0's binary_logloss: 0.606701
Early stopping, best iteration is:
[27]	valid_0's average_precision: 0.358295	valid_0's binary_logloss: 0.609868
Evaluated only: average_precision
Best iteration: 27


In [101]:
# ============================================================
# PREDIKSI PROBABILITAS CANDIDATE V3
# ============================================================

best_iteration_v3 = (
    lightgbm_v3_model.best_iteration_
)

train_probabilities_v3 = (
    lightgbm_v3_model.predict_proba(
        X_train_v3,
        num_iteration=best_iteration_v3,
    )[:, 1]
)

validation_probabilities_v3 = (
    lightgbm_v3_model.predict_proba(
        X_validation_v3,
        num_iteration=best_iteration_v3,
    )[:, 1]
)

calibration_probabilities_v3 = (
    lightgbm_v3_model.predict_proba(
        X_calibration_v3,
        num_iteration=best_iteration_v3,
    )[:, 1]
)

test_probabilities_v3 = (
    lightgbm_v3_model.predict_proba(
        X_test_v3,
        num_iteration=best_iteration_v3,
    )[:, 1]
)

pr_auc_summary_v3 = pd.DataFrame([
    {
        "split": "train",
        "pr_auc": average_precision_score(
            y_train_v3,
            train_probabilities_v3,
        ),
    },
    {
        "split": "validation",
        "pr_auc": average_precision_score(
            y_validation_v3,
            validation_probabilities_v3,
        ),
    },
    {
        "split": "calibration",
        "pr_auc": average_precision_score(
            y_calibration_v3,
            calibration_probabilities_v3,
        ),
    },
    {
        "split": "test",
        "pr_auc": average_precision_score(
            y_test_v3,
            test_probabilities_v3,
        ),
    },
])

display(
    pr_auc_summary_v3
)

,split,pr_auc
0,train,0.459213
1,validation,0.358295
2,calibration,0.354360
3,test,0.348577


In [102]:
# ============================================================
# PILIH THRESHOLD DARI CALIBRATION SET
# ============================================================

(
    calibration_summary_v3,
    calibration_operating_points_v3,
) = select_threshold_v3(
    model_name=(
        "LightGBM Candidate V3 Calibration"
    ),
    y_true=y_calibration_v3,
    probabilities=(
        calibration_probabilities_v3
    ),
)

selected_threshold_v3 = float(
    calibration_summary_v3
    .iloc[0]["threshold"]
)

print(
    "Selected threshold V3:",
    selected_threshold_v3,
)

display(
    calibration_summary_v3
)

Selected threshold V3: 0.25393263422131074


,model,status,selection_strategy,pr_auc,threshold,precision,recall,f1,false_positive_rate,true_positive,false_positive,false_negative,true_negative
0,LightGBM Candidate V3 Calibration,TARGET NOT ACHIEVED,RECALL GUARDRAIL,0.35436,0.253933,0.308502,0.999645,0.471495,0.993447,16909,37901,6,250


In [103]:
# ============================================================
# EVALUASI DENGAN THRESHOLD YANG SUDAH DIKUNCI
# ============================================================

validation_summary_v3 = (
    evaluate_fixed_threshold_v3(
        model_name=(
            "LightGBM Candidate V3 Validation"
        ),
        y_true=y_validation_v3,
        probabilities=(
            validation_probabilities_v3
        ),
        threshold=selected_threshold_v3,
    )
)

test_summary_v3 = (
    evaluate_fixed_threshold_v3(
        model_name=(
            "LightGBM Candidate V3 Final Test"
        ),
        y_true=y_test_v3,
        probabilities=(
            test_probabilities_v3
        ),
        threshold=selected_threshold_v3,
    )
)

final_evaluation_summary_v3 = pd.concat(
    [
        calibration_summary_v3,
        validation_summary_v3,
        test_summary_v3,
    ],
    ignore_index=True,
    sort=False,
)

display(
    final_evaluation_summary_v3
)

,model,status,selection_strategy,pr_auc,threshold,precision,recall,f1,false_positive_rate,true_positive,false_positive,false_negative,true_negative,rows,fraud_rate,precision_lift,alert_rate
0,LightGBM Candidate V3 Calibration,TARGET NOT ACHIEVED,RECALL GUARDRAIL,0.354360,0.253933,0.308502,0.999645,0.471495,0.993447,16909,37901,6,250,NaN,NaN,NaN,NaN
1,LightGBM Candidate V3 Validation,TARGET NOT ACHIEVED,NaN,0.358295,0.253933,0.309183,0.999616,0.472287,0.999508,18201,40667,7,20,58895.0,0.309160,1.000074,0.999542
2,LightGBM Candidate V3 Final Test,TARGET NOT ACHIEVED,NaN,0.348577,0.253933,0.307330,0.998952,0.470048,0.897196,18118,40835,19,4679,63651.0,0.284944,1.078560,0.926191


In [104]:
# ============================================================
# TEST PREDICTION DIAGNOSTIC
# ============================================================

test_predictions_v3 = (
    test_probabilities_v3
    >= selected_threshold_v3
).astype(np.int8)

tn_v3, fp_v3, fn_v3, tp_v3 = (
    confusion_matrix(
        y_test_v3,
        test_predictions_v3,
        labels=[0, 1],
    )
    .ravel()
)

test_confusion_matrix_v3 = pd.DataFrame(
    [
        {
            "actual_class": "NORMAL",
            "predicted_normal": int(tn_v3),
            "predicted_fraud": int(fp_v3),
        },
        {
            "actual_class": "FRAUD",
            "predicted_normal": int(fn_v3),
            "predicted_fraud": int(tp_v3),
        },
    ]
)

display(
    test_confusion_matrix_v3
)

print(
    classification_report(
        y_test_v3,
        test_predictions_v3,
        labels=[0, 1],
        target_names=[
            "normal",
            "fraud",
        ],
        digits=6,
        zero_division=0,
    )
)

,actual_class,predicted_normal,predicted_fraud
0,NORMAL,4679,40835
1,FRAUD,19,18118


              precision    recall  f1-score   support

      normal   0.995956  0.102804  0.186370     45514
       fraud   0.307330  0.998952  0.470048     18137

    accuracy                       0.358156     63651
   macro avg   0.651643  0.550878  0.328209     63651
weighted avg   0.799736  0.358156  0.267202     63651



In [105]:
# ============================================================
# FEATURE IMPORTANCE CANDIDATE V3
# ============================================================

feature_importance_v3 = pd.DataFrame({
    "feature": (
        lightgbm_v3_model
        .booster_
        .feature_name()
    ),

    "importance_gain": (
        lightgbm_v3_model
        .booster_
        .feature_importance(
            importance_type="gain"
        )
    ),

    "importance_split": (
        lightgbm_v3_model
        .booster_
        .feature_importance(
            importance_type="split"
        )
    ),
})

total_gain_v3 = (
    feature_importance_v3[
        "importance_gain"
    ].sum()
)

feature_importance_v3[
    "gain_percentage"
] = np.where(
    total_gain_v3 > 0,
    (
        feature_importance_v3[
            "importance_gain"
        ]
        / total_gain_v3
        * 100
    ),
    0,
)

feature_importance_v3 = (
    feature_importance_v3
    .sort_values(
        "importance_gain",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    feature_importance_v3.head(25)
)

print(
    "Total fitur dengan importance nol:",
    int(
        (
            feature_importance_v3[
                "importance_gain"
            ] == 0
        ).sum()
    ),
)

,feature,importance_gain,importance_split,gain_percentage
0,initiator_amount_max_prior,108861.641503,35,31.160890
1,amount,61494.696842,64,17.602430
2,new_bal_initiator,59361.570822,56,16.991838
3,initiator_amount_mean_prior,38727.495947,31,11.085477
4,initiator_new_balance_negative,10517.517914,6,3.010567
5,initiator_balance_change,8741.839644,30,2.502291
6,amount_vs_initiator_mean,8438.954040,27,2.415592
7,amount_vs_initiator_max,8129.789847,32,2.327096
8,amount_log1p,7366.050537,6,2.108481
9,recipient_tx_count_prior,6190.411116,120,1.771962


Total fitur dengan importance nol: 7
